In [5]:
import sys
import os

# For notebooks, use getcwd(); for scripts, use __file__
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Running in a notebook
    notebook_dir = os.getcwd()

sys.path.insert(0, notebook_dir)

from agents.orchestrator import analyze_user_input, update_state_from_analysis
from agents.specialist import run_activities_specialist, run_logistics_specialist
from agents.verifier import verify_and_format_itinerary, format_complete_itinerary
from state import TravelState


In [6]:
state = TravelState()
task_json ={
    "task_id": "ORCH-T1-02",
    "tier": 1,
    "difficulty": "easy",
    "description": "Round-trip flight with specific dates",
    "category": "orchestrator",
    "input": {
      "type": "natural_language",
      "format": "text",
      "content": "Find the cheapest round-trip flight from Boston to Seattle, departing May 5 and returning May 12, 2026."
    }
    }
user_input = None #"want to change my trip to Seattle from May 6th to May 13th"

analysis = analyze_user_input(user_input, state, task_json=task_json)
state = update_state_from_analysis(state, analysis)
delegation = analysis.get("delegation", "none")

In [7]:
analysis

{'intent': 'new_trip',
 'updated_constraints': {'origin': 'Boston',
  'destination': 'Seattle',
  'start_date': '2026-05-05',
  'end_date': '2026-05-12',
  'num_travelers': 1},
 'delegation': 'logistics',
 'response_to_user': "I'll help you find the cheapest round-trip flight from Boston to Seattle, departing May 5 and returning May 12, 2026. Let me search for the best flight options for you."}

In [8]:
state

TravelState(task_id='ORCH-T1-02', constraints=TravelConstraints(origin='Boston', destination='Seattle', start_date='2026-05-05', end_date='2026-05-12', budget_usd=None, num_travelers=1, dietary_needs=[], interests=[], preferred_hotel_rating=None), draft_itinerary=None, delegation_plan='logistics', messages=[], last_verification=None)

In [9]:
 # 3. Delegation
if delegation == "none":
    response = analysis.get(
        "response_to_user",
        "I'm still missing some info to start planning.",
    )
    print(f"\nNomad: {response}")
    state.messages.append({"role": "assistant", "content": response})

In [10]:

# If we reach here, we need to run specialists
draft_components = []
all_search_results = {
    "flights": [],
    "hotels": [],
    "activities": [],
}
# Accumulate specialist contexts for verifier
specialist_contexts = {
    "logistics": None,
    "activities": None,
}
constraints_str = state.constraints.model_dump_json(indent=2)

if delegation in ["logistics", "both"]:
    print("\n[Specialist - Logistics] Searching for flights & hotels...")
    try:
        # NEW: Unpack all 3 return values
        logistics_draft, logistics_searches, logistics_context = run_logistics_specialist(
            constraints_str, 
            task_id=state.task_id
        )
        draft_components.append("--- LOGISTICS ---\n" + logistics_draft)
        
        # Accumulate search results
        all_search_results["flights"].extend(logistics_searches.get("flights", []))
        all_search_results["hotels"].extend(logistics_searches.get("hotels", []))
        
        # Save specialist context for verifier
        specialist_contexts["logistics"] = logistics_context
        
        print(f"\n[Logistics Specialist Summary]")
        print(f"  Search Coverage: {logistics_context['search_coverage']}")
        print(f"  Total Searches: {logistics_context['total_searches']}")
        print(f"  Tool Calls: {len(logistics_context['tool_calls_summary'])}")
        
    except Exception as e:
        print(f"[Specialist Error] {e}")
        import traceback
        traceback.print_exc()



[Specialist - Logistics] Searching for flights & hotels...
  [Specialist Turn 1] Thinking...
  🔧 Executing tool search_flights with arguments {'origin': 'BOS', 'destination': 'SEA', 'departure_date': '2026-05-05', 'return_date': '2026-05-12', 'task_id': 'ORCH-T1-02', 'turn': 0}
  [Cache Hit] google_flights
✈️  Flights results: 3 options
[Saved] Candidate -> c:\GIT\aiagent\nomad\src\output\search_candidates\ORCH-T1-02\flights_candidates.json (1 total)
  🔧 Executing tool search_hotels with arguments {'location': 'Seattle, Washington', 'check_in': '2026-05-05', 'check_out': '2026-05-12', 'adults': 1, 'task_id': 'ORCH-T1-02', 'turn': 0}
  [API Call] google_hotels...
  [Saved] google_hotels -> c:\GIT\aiagent\nomad\src\output\cache
🏨  Hotels results: 5 options
[Saved] Candidate -> c:\GIT\aiagent\nomad\src\output\search_candidates\ORCH-T1-02\hotels_candidates.json (1 total)
  [Specialist Turn 2] Thinking...

[Logistics Specialist Summary]
  Search Coverage: {'flights': 6, 'hotels': 5, 'activ

In [11]:
print("\n[Verifier] Checking itinerary against constraints...")
full_draft = "\n\n".join(draft_components)

print(f"\n[Debug] Search Results Found:")
print(f"  - Flights: {len(all_search_results['flights'])} items")
print(f"  - Hotels: {len(all_search_results['hotels'])} items")
print(f"  - Activities: {len(all_search_results['activities'])} items")

try:
    verification = verify_and_format_itinerary(
        full_draft, 
        constraints_str,
        task_id=state.task_id,
        search_results=all_search_results  # Pass search results to verifier
    )

    if verification.get("is_valid"):
        final_response = verification.get(
            "final_message_to_user", "Here is your plan!"
        )
        print("\nNomad:\n")
        print(final_response)
        state.messages.append({"role": "assistant", "content": final_response})
    else:
        issues = verification.get("issues", [])
        print(
            f"\nNomad: I built a plan, but it violates some constraints:\n{issues}"
        )
        print("I will need to revise. Let me know how you'd like to adjust.")

except Exception as e:
    print(f"Error during verification: {e}")


[Verifier] Checking itinerary against constraints...

[Debug] Search Results Found:
  - Flights: 7 items
  - Hotels: 6 items
  - Activities: 0 items
[Saved] Verification result -> c:\GIT\aiagent\nomad\src\output\verification_results\ORCH-T1-02_verification.json

Nomad:

🎉 Your Seattle adventure is confirmed!

**Trip Summary: Boston → Seattle (May 5-12, 2026)**

✈️ **Outbound Flight - Delta DL 384**
- Departs: May 5 at 7:37 AM from Boston Logan (BOS)
- Arrives: May 5 at 10:55 AM PST at Seattle-Tacoma (SEA)
- Duration: 6h 18m NONSTOP
- Amenities: Free Wi-Fi, power outlets, Live TV
- Eco-friendly: 27% less carbon than typical flights

🏨 **Accommodation - Green Tortoise Hostel Seattle**
- Check-in: May 5, 2026 (2:00 PM)
- Check-out: May 12, 2026 (11:00 AM)
- 7 nights total
- Rating: 4.6/5 ⭐ (2,528 reviews)
- Location: ⭐⭐⭐⭐⭐ 4.9/5 - Just 1 minute walk to Pike Place Market!
- Amenities: Free breakfast, Free Wi-Fi, communal kitchen, air conditioning

💰 **Total Trip Cost: $676**
- Flight: $29

In [14]:
# 5. Save Agent Result to Repository (Independent)
# ===================================================
# 保存完整的 verification 结果（含 constraints），方便后续评估
from tools.plan_repository import PlanRepository

print("\n" + "="*70)
print("[SAVE] Saving agent output to repository...")
print("="*70)

if verification.get("is_valid"):
    plan_to_save = verification.get("itinerary", verification)
    
    # Save to repository (uses config.PLANS_DIR by default)
    repo = PlanRepository()
    saved_path = repo.save_plan(
        plan=plan_to_save, 
        task_id=state.task_id,
        save_metadata=True
    )
    
    print(f"✓ Plan saved to: {saved_path}")
    
    # Show repository status
    all_plans = repo.get_all_plans()
    print(f"\n[REPOSITORY STATUS]")
    print(f"  Total plans saved: {len(all_plans)}")
    print(f"  Task IDs: {all_plans}")
    
else:
    print(f"✗ Cannot save invalid plan")
    print(f"  Issues: {verification.get('issues', [])}")


[SAVE] Saving agent output to repository...
✓ Plan saved: c:\GIT\aiagent\nomad\src\output\plans\ORCH-T1-02\plan.json
✓ Plan saved to: c:\GIT\aiagent\nomad\src\output\plans\ORCH-T1-02\plan.json

[REPOSITORY STATUS]
  Total plans saved: 2
  Task IDs: ['ORCH-T1-02', 'task_123']


In [15]:
# 6. BATCH EVALUATOR - Evaluate All Saved Plans at Once
# =========================================================
# 统一评估所有之前保存的 plans
import importlib
import tools.plan_schema, tools.evaluator, tools.auto_evaluator
importlib.reload(tools.plan_schema)
importlib.reload(tools.evaluator)
importlib.reload(tools.auto_evaluator)
from tools.auto_evaluator import AutoEvaluator
import json
from pathlib import Path

print("\n" + "="*70)
print("[BATCH EVALUATOR] Evaluating all saved plans from repository")
print("="*70)

# Initialize auto evaluator (uses config.PLANS_DIR and config.EVALUATIONS_DIR by default)
auto_eval = AutoEvaluator()

# Show what we're about to evaluate
repo_plans = auto_eval.repo.get_all_plans()
print(f"\nFound {len(repo_plans)} plans to evaluate:")
for task_id in repo_plans:
    print(f"   - {task_id}")

print("\nEvaluating...")

# One-line evaluation: read all plans -> evaluate -> generate report
results, report_file = auto_eval.auto_evaluate_and_report(report_name="batch_evaluation")

print(f"\n[DONE] Report saved to: {report_file}")

# Display statistics
if results:
    scores = [r['overall_score'] for r in results.values() if 'overall_score' in r]
    
    if scores:
        print(f"\n[SUMMARY]")
        print(f"  Total evaluated: {len(scores)}")
        print(f"  Average score: {sum(scores)/len(scores):.1%}")
        print(f"  Best score: {max(scores):.1%}")
        print(f"  Worst score: {min(scores):.1%}")
    
    excellent = sum(1 for s in scores if s >= 0.8)
    acceptable = sum(1 for s in scores if 0.6 <= s < 0.8)
    needs_revision = sum(1 for s in scores if s < 0.6)
    
    print(f"\n[RATINGS]")
    print(f"  Excellent (>=0.8): {excellent}")
    print(f"  Acceptable (0.6-0.8): {acceptable}")
    print(f"  Needs Revision (<0.6): {needs_revision}")
    
    # Detail per plan
    for task_id, result in results.items():
        if 'overall_score' in result:
            print(f"\n[{task_id}]")
            print(f"  Overall: {result['overall_score']:.1%}")
            print(f"  Schema:  {result.get('schema_compliance', 0):.1%}")
            print(f"  CSR:     {result.get('csr_score', 0):.1%}")
            print(f"  Tools:   {result.get('tool_accuracy', 0):.1%}")
            if result.get('constraint_breakdown'):
                for c, met in result['constraint_breakdown'].items():
                    print(f"    {'Y' if met else 'N'} {c}")
            if result.get('conflict_report', {}).get('warnings'):
                for w in result['conflict_report']['warnings']:
                    print(f"    ! {w}")


[BATCH EVALUATOR] Evaluating all saved plans from repository

Found 2 plans to evaluate:
   - ORCH-T1-02
   - task_123

Evaluating...

[AutoEvaluator] Evaluating 2 plans...
[AutoEvaluator] Evaluating ORCH-T1-02... ✓ Loaded plan from repository: ORCH-T1-02
✓ Score: 91.0%
[AutoEvaluator] Evaluating task_123... ✓ Loaded plan from repository: task_123
✓ Score: 75.0%

EVALUATION SUMMARY
  ✓ EXCELLENT     ORCH-T1-02            91.0%
  ~ ACCEPTABLE    task_123              75.0%

----------------------------------------------------------------------
TOTALS                             2
  Excellent:     1
  Acceptable:    1
  Needs Revision: 0

Average Score: 83.0%
Best Score:    91.0%
Worst Score:   75.0%

✓ Report saved: c:\GIT\aiagent\nomad\src\output\evaluations\batch_evaluation.json

[DONE] Report saved to: c:\GIT\aiagent\nomad\src\output\evaluations\batch_evaluation.json

[SUMMARY]
  Total evaluated: 2
  Average score: 83.0%
  Best score: 91.0%
  Worst score: 75.0%

[RATINGS]
  Excellen